In [6]:
from torch import optim
from ReplayBuffer import ReplayBuffer
from DQN import DQN
from utils import (
    get_numeric_state,
    choose_action,
    apply_action,
    get_reward,
    train_step_batch,
)
import traci


sumo_binary = "sumo-gui"
sumo_config = "./scenarios/bologna/run.sumocfg"

view_id = "View #0"

car_id = "Borgo_100_0"
control_enabled = False






In [7]:
policy_net = DQN(state_dim=3, action_dim=3)
target_net = DQN(state_dim=3, action_dim=3)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
replay_buffer = ReplayBuffer(capacity=10000)
batch_size = 32

epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.05
target_update_freq = 10

control_enabled = False
prev_state = None

In [11]:
for step in range(300):
    traci.simulationStep()

    if car_id not in traci.vehicle.getIDList():
        print(f"step={step}: vehicle not in sim")
        continue

    if not control_enabled:
        traci.vehicle.setSpeedMode(car_id, 0)
        control_enabled = True

    state = get_numeric_state(car_id)

    action = choose_action(policy_net, state, epsilon)
    apply_action(car_id, action)

    # advance once more so action has consequence in the sim
    traci.simulationStep()

    if car_id not in traci.vehicle.getIDList():
        print(f"step={step}: vehicle disappeared after action")

        next_state = state
        reward = -20.0
        done = 1.0

        replay_buffer.push(state, int(action), reward, next_state, done)
        break

    next_state = get_numeric_state(car_id)
    reward = get_reward(car_id)
    done = 0.0

    replay_buffer.push(state, int(action), reward, next_state, done)

    loss = train_step_batch(
        policy_net=policy_net,
        target_net=target_net,
        optimizer=optimizer,
        replay_buffer=replay_buffer,
        batch_size=batch_size,
        gamma=0.99,
    )

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if step % target_update_freq == 0:
        target_net.load_state_dict(policy_net.state_dict())

    loss_str = f"{loss:8.5f}" if loss is not None else " warmup "

    print(
        f"step={step:03d} "
        f"action={action.name:<7} "
        f"state={state} "
        f"reward={reward:6.3f} "
        f"loss={loss_str} "
        f"epsilon={epsilon:5.3f}"
    )

traci.close()

 Retrying in 1 seconds
step=000 action=FASTER  state=[0.0, 1.0, 0.0] reward= 0.200 loss= warmup  epsilon=0.050
step=001 action=SLOWER  state=[0.1, 1.0, 0.0] reward= 0.000 loss= warmup  epsilon=0.050
step=002 action=SLOWER  state=[0.0, 1.0, 0.0] reward= 0.000 loss= warmup  epsilon=0.050
step=003 action=SLOWER  state=[0.0, 1.0, 0.0] reward= 0.000 loss= warmup  epsilon=0.050
step=004 action=FASTER  state=[0.0, 1.0, 0.0] reward= 0.200 loss= warmup  epsilon=0.050
step=005 action=SLOWER  state=[0.1, 1.0, 0.0] reward= 0.000 loss= warmup  epsilon=0.050
step=006 action=SLOWER  state=[0.0, 1.0, 0.0] reward= 0.000 loss= warmup  epsilon=0.050
step=007 action=SLOWER  state=[0.0, 1.0, 0.0] reward= 0.000 loss= warmup  epsilon=0.050
step=008 action=SLOWER  state=[0.0, 1.0, 0.0] reward= 0.000 loss= warmup  epsilon=0.050
step=009 action=FASTER  state=[0.0, 1.0, 0.0] reward= 0.200 loss= warmup  epsilon=0.050
step=010 action=SLOWER  state=[0.1, 1.0, 0.0] reward= 0.000 loss= warmup  epsilon=0.050
step=011 